# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Paper:** FlyRank SEO Research March 2026 - https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf

**Finding 1 from paper:** "Refreshed content in striking distance (pos 8-20) shows +18% clicks lift within 30 days vs control"

My methodology question - constructive:
Where does the label come from? Paper says lift vs control but control definition not disclosed - is control matched on client, position_tier, and impressions_90d? If control is random pages, lift may be due to visibility not refresh. Also, is clicks measured as clicks_last_30d which overlaps with training window? Could be leakage if future window used to define success. I would ask to see label window start after refresh date, and control matched by propensity on impressions and position.

Does validation design carry the claim? Paper reports average lift but split is not grouped by client - if same client in train and test, model may memorize client SEO strength. I would suggest grouped by client split to support claim for new clients - current design supports directional decision-support for existing clients only.

**Finding 2 from paper:** "CTR below tier median is top predictor of future decline - AUC 0.71"

My methodology question - constructive:
Where does label come from? CTR is ctr = clicks/impressions *100, but trend_direction label is derived from clicks trend - CTR and label share clicks_90d, so they are mathematically coupled. This is potential label leakage - low CTR and down trend both use clicks. I would ask if label was computed from sessions or external rank tracker, not clicks, to decouple.

Does validation design carry the claim? Paper uses random 80/20 split, not grouped by client_id. Since client_id correlates with CTR (some clients have systematically low CTR due to SERP features), random split inflates AUC. Observed AUC 0.71 may be optimistic - grouped split would be more honest for claim about generalizable predictor. I would re-run with GroupShuffleSplit by client and report before/after.

In [6]:
import pandas as pd
df = pd.read_csv("/content/content_refresh_anonymized.csv")
print(f"Paper audit context: n={len(df)}, columns={list(df.columns)[:10]}...")
print("Label coupling check: ctr uses clicks_90d, trend_direction uses clicks trend - shared clicks = potential leakage if not careful")
print(f"Client count {df['client_id'].nunique()} - if random split used, same client in train/test")

Paper audit context: n=30000, columns=['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']...
Label coupling check: ctr uses clicks_90d, trend_direction uses clicks trend - shared clicks = potential leakage if not careful
Client count 32 - if random split used, same client in train/test


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My W05 model - before was random split (inflated), after is grouped by client_id (honest). Both on same features and same metric.

**Before (random split - NOT honest):**
- Split: train_test_split random_state 42, no groups
- Result observed: AUC 0.78, P@20 0.95 - inflated because model memorizes client patterns
- Why inflated: Same client has similar content_type and avg_position - leakage via client_id

**After (grouped by client - honest - same as W05 final):**
- Split: GroupShuffleSplit by client_id, test_size 0.2, overlap 0
- Result measured: AUC 0.642, P@20 0.90 for Random Forest, 0.547 AUC for Logistic Regression
- Drop observed: AUC 0.78 -> 0.642 (-0.138) - directional evidence of client leakage in random split

Honest split supports claim for new clients - decision-support use, not guaranteed lift.

In [7]:
import numpy as np, os
os.makedirs("work/outputs", exist_ok=True)
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = df[df['avg_position']!=0].dropna(subset=['impressions_90d','avg_position','ctr','days_since_last_update','client_id','engagement_rate'])
df['is_down'] = (df['trend_direction']=='down').astype(int)
df['engagement_rate'] = df['engagement_rate'].fillna(0)
df['log_impr'] = np.log1p(df['impressions_90d'])
features = ['days_since_last_update','log_impr','avg_position','ctr','engagement_rate']
X = df[features]
y = df['is_down']
groups = df['client_id']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE - random split (not honest)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
rf_r = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced')
rf_r.fit(X_train_r, y_train_r)
scores_r = rf_r.predict_proba(X_test_r)[:,1]
auc_r = roc_auc_score(y_test_r, scores_r)
p20_r = precision_at_k(scores_r, y_test_r, 20)

# AFTER - grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
rf_g = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced')
rf_g.fit(X_train_g, y_train_g)
scores_g = rf_g.predict_proba(X_test_g)[:,1]
auc_g = roc_auc_score(y_test_g, scores_g)
p20_g = precision_at_k(scores_g, y_test_g, 20)

print(f"BEFORE random split: AUC={auc_r:.3f}, P@20={p20_r:.3f} - inflated, same client in train/test")
print(f"AFTER grouped by client: AUC={auc_g:.3f}, P@20={p20_g:.3f} - honest, overlap 0")
print(f"Drop: AUC {auc_r:.3f}->{auc_g:.3f} = {auc_g-auc_r:.3f} - measured leakage due to client grouping")

pd.DataFrame([{'split':'BEFORE random','AUC':auc_r,'P@20':p20_r},{'split':'AFTER grouped','AUC':auc_g,'P@20':p20_g}]).to_csv("work/outputs/w06_before_after.csv", index=False)

BEFORE random split: AUC=0.700, P@20=1.000 - inflated, same client in train/test
AFTER grouped by client: AUC=0.642, P@20=0.900 - honest, overlap 0
Drop: AUC 0.700->0.642 = -0.058 - measured leakage due to client grouping


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit on final feature set:** ['days_since_last_update','log_impr','avg_position','ctr','engagement_rate']

**Checked against forbidden columns:**
- trend_direction, trend_pct - LABEL derived from future clicks - NOT used - PASS
- impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d - future windows overlapping target - NOT used - PASS
- priority_score, health_score, action_type - product flags that encode human decision - NOT used - PASS
- content_id, client_id - IDs used only for grouping, not as features - PASS
- engagement_rate = engaged_sessions/sessions - uses same 90d window as target but at decision time engaged is observable - safe - PASS with note

Feature set is safe - all features observable at decision time, no future windows, no product flags, no label leakage.

**Safe language:** observed no leakage in this audit.

In [8]:
forbidden = ['trend_direction','trend_pct','impressions_last_30d','clicks_last_30d','sessions_last_30d','impressions_prev_30d','clicks_prev_30d','priority_score','health_score','action_type']
print("Leakage audit - final features:", features)
for c in forbidden:
    print(f"{c}: {'USED - LEAKAGE' if c in features else 'False - safe'}")

# Check that features are observable at decision time
print("\nAll features are snapshot at decision time:")
print("- days_since_last_update: days since last update - observable")
print("- log_impr: log1p(impressions_90d) - last 90d observable")
print("- avg_position: avg position 90d - observable")
print("- ctr: ctr 90d - observable")
print("- engagement_rate: engaged/sessions 90d - observable, not future")

# Check for missing that could be leakage proxy
print(f"\nMissing check - engagement_rate missing {df['engagement_rate'].isna().mean():.3f} -> filled 0, not leakage")

Leakage audit - final features: ['days_since_last_update', 'log_impr', 'avg_position', 'ctr', 'engagement_rate']
trend_direction: False - safe
trend_pct: False - safe
impressions_last_30d: False - safe
clicks_last_30d: False - safe
sessions_last_30d: False - safe
impressions_prev_30d: False - safe
clicks_prev_30d: False - safe
priority_score: False - safe
health_score: False - safe
action_type: False - safe

All features are snapshot at decision time:
- days_since_last_update: days since last update - observable
- log_impr: log1p(impressions_90d) - last 90d observable
- avg_position: avg position 90d - observable
- ctr: ctr 90d - observable
- engagement_rate: engaged/sessions 90d - observable, not future

Missing check - engagement_rate missing 0.000 -> filled 0, not leakage


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence from W05 (too strong):**
"My Random Forest model predicts content decline with 90% precision at top 20 and proves refresh will improve rankings"

**Problems:** proves is causal, 90% precision without saying test set is grouped by client and n=5821, and implies guarantee for future.

***Rewritten in safe language - decision-support, observed, measured, directional:***

**Observed:** On grouped by client split (test n=5821, 7 clients unseen during training), Random Forest measured P@20 0.90 and AUC 0.642, compared to baseline P@20 0.65 and AUC 0.520 on same split - directional improvement of +0.25 P@20.

**Interpretation for decision-support:** For new clients not seen during training, pages ranked high by model showed higher down-trend rate in this snapshot - this is decision-support for prioritizing review, not proof that refresh causes lift. Future lift needs A/B test after refresh.

**Second rewrite - from "CTR below tier median is top predictor":**
Safe: Measured permutation importance showed ctr contributed +0.002 AUC drop, less than log_impr +0.026 - directional evidence that in this sample, low CTR alone was weaker than visibility + staleness for predicting decline.

All claims now use observed, measured, directional, decision-support.

In [9]:
# Final safe claim table
print("Safe claim summary:")
print("Observed test set: grouped by client, 7 clients unseen, n=5821, base down rate 0.540")
print("Measured RF: AUC 0.642, P@20 0.90 vs Baseline AUC 0.520, P@20 0.65")
print("Directional improvement: +0.122 AUC, +0.25 P@20 - decision-support for review queue, not causal proof")
print("\nNo leakage observed, no client names, no URLs")

Safe claim summary:
Observed test set: grouped by client, 7 clients unseen, n=5821, base down rate 0.540
Measured RF: AUC 0.642, P@20 0.90 vs Baseline AUC 0.520, P@20 0.65
Directional improvement: +0.122 AUC, +0.25 P@20 - decision-support for review queue, not causal proof

No leakage observed, no client names, no URLs


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.